# IS 362 – Week 5 Assignment
**Bria King**

This notebook uses Python and pandas to analyze airport and weather data
from the nycflights13 repository.

The analysis answers three questions:
1. What is the northernmost airport in the United States?
2. What is the easternmost airport in the United States?
3. Which New York area airport had the windiest weather on February 12, 2013?

Airport locations will be checked against additional sources. Weather
observations will be examined for outliers before drawing conclusions.
Assumptions and any data changes will be documented.

In [1]:
import pandas as pd

base_url = "https://raw.githubusercontent.com/hadley/nycflights13/master/data-raw/"

airports = pd.read_csv(base_url + "airports.csv")
weather = pd.read_csv(base_url + "weather.csv")

print("Airport data:", airports.shape)
print("Weather data:", weather.shape)

display(airports.head())
display(weather.head())

Airport data: (1458, 8)
Weather data: (26115, 15)


,faa,name,lat,lon,alt,tz,dst,tzone
0,04G,Lansdowne Airport,41.130472,-80.619583,1044,-5,A,America/New_York
1,06A,Moton Field Municipal Airport,32.460572,-85.680028,264,-6,A,America/Chicago
2,06C,Schaumburg Regional,41.989341,-88.101243,801,-6,A,America/Chicago
3,06N,Randall Airport,41.431912,-74.391561,523,-5,A,America/New_York
4,09J,Jekyll Island Airport,31.074472,-81.427778,11,-5,A,America/New_York


,origin,year,month,day,hour,temp,dewp,humid,wind_dir,wind_speed,wind_gust,precip,pressure,visib,time_hour
0,EWR,2013,1,1,1,39.02,26.06,59.37,270.0,10.35702,NaN,0.0,1012.0,10.0,2013-01-01T06:00:00Z
1,EWR,2013,1,1,2,39.02,26.96,61.63,250.0,8.05546,NaN,0.0,1012.3,10.0,2013-01-01T07:00:00Z
2,EWR,2013,1,1,3,39.02,28.04,64.43,240.0,11.50780,NaN,0.0,1012.5,10.0,2013-01-01T08:00:00Z
3,EWR,2013,1,1,4,39.92,28.04,62.21,250.0,12.65858,NaN,0.0,1012.2,10.0,2013-01-01T09:00:00Z
4,EWR,2013,1,1,5,39.02,28.04,64.43,260.0,12.65858,NaN,0.0,1011.9,10.0,2013-01-01T10:00:00Z


In [2]:
# Examine the five airports with the highest latitude.
northern_candidates = airports.nlargest(5, "lat")

display(
    northern_candidates[["faa", "name", "lat", "lon", "tzone"]]
)

,faa,name,lat,lon,tzone
417,EEN,Dillant Hopkins Airport,72.270833,42.898333,NaN
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003,America/Anchorage
110,AIN,Wainwright Airport,70.638056,-159.994722,America/Anchorage
708,K03,Wainwright As,70.613378,-159.860350,America/Anchorage
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000,America/Anchorage


In [3]:
# Preserve the original data and correct EEN's coordinates
# using FAA information published by AirNav.
airports_clean = airports.copy()

airports_clean.loc[
    airports_clean["faa"] == "EEN", ["lat", "lon"]
] = [42.8983889, -72.2707778]

display(
    airports_clean.nlargest(5, "lat")[
        ["faa", "name", "lat", "lon", "tzone"]
    ]
)

,faa,name,lat,lon,tzone
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003,America/Anchorage
110,AIN,Wainwright Airport,70.638056,-159.994722,America/Anchorage
708,K03,Wainwright As,70.613378,-159.860350,America/Anchorage
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000,America/Anchorage
1363,UUK,Ugnu-Kuparuk Airport,70.330833,-149.597500,America/Anchorage


## Question 1: Northernmost Airport — Data Validation

The original ranking placed Dillant/Hopkins Airport (EEN) first,
with latitude 72.270833 and longitude 42.898333. These coordinates
conflict with its actual location in Keene, New Hampshire.

AirNav's published FAA information lists its coordinates as
42.8983889 latitude and -72.2707778 longitude. I corrected these
two values in a copy named `airports_clean`, preserving the original
dataset.

After this correction, Wiley Post–Will Rogers Memorial Airport
(BRW) has the highest latitude in the dataset: 71.285446° N.
This is the leading candidate, pending external verification.

Source: https://www.airnav.com/airport/KEEN

Scope: This ranking compares airports included in the supplied CSV;
the dataset may not include every U.S. airport.

In [4]:
# Examine the five largest latitude values.
eastern_candidates = airports_clean.nlargest(5, "lon")

display(
    eastern_candidates[["faa", "name", "lat", "lon", "tzone"]]
)

,faa,name,lat,lon,tzone
1290,SYA,Eareckson As,52.712275,174.113620,America/Anchorage
942,MYF,Montgomery Field,32.475900,117.759000,Asia/Chongqing
396,DVT,Deer Valley Municipal Airport,33.411700,112.457000,Asia/Chongqing
444,EPM,Eastport Municipal Airport,44.910111,-67.012694,America/New_York
624,HUL,Houlton Intl,46.123083,-67.792056,America/New_York


In [5]:
# Inspect records requiring geographic validation.
display(
    airports_clean.loc[
        airports_clean["faa"].isin(["SYA", "MYF", "DVT"])
    ]
)

,faa,name,lat,lon,alt,tz,dst,tzone
396,DVT,Deer Valley Municipal Airport,33.411700,112.45700,1478,8,A,Asia/Chongqing
942,MYF,Montgomery Field,32.475900,117.75900,17,8,A,Asia/Chongqing
1290,SYA,Eareckson As,52.712275,174.11362,98,-9,A,America/Anchorage


In [6]:
# Correct coordinates using FAA information published by AirNav.
# Sources:
# https://www.airnav.com/airport/MYF
# https://www.airnav.com/airport/DVT

airports_clean.loc[
    airports_clean["faa"] == "MYF", ["lat", "lon"]
] = [32.8157222, -117.1395556]

airports_clean.loc[
    airports_clean["faa"] == "DVT", ["lat", "lon"]
] = [33.6883056, -112.0825556]

# Rank the corrected data by signed longitude.
display(
    airports_clean.nlargest(5, "lon")[
        ["faa", "name", "lat", "lon", "tzone"]
    ]
)

,faa,name,lat,lon,tzone
1290,SYA,Eareckson As,52.712275,174.113620,America/Anchorage
444,EPM,Eastport Municipal Airport,44.910111,-67.012694,America/New_York
624,HUL,Houlton Intl,46.123083,-67.792056,America/New_York
259,CAR,Caribou Muni,46.871500,-68.017917,America/New_York
1101,PQI,Northern Maine Rgnl At Presque Isle,46.688958,-68.044797,America/New_York


## Question 2: Easternmost Airport — Data Validation

MYF and DVT originally had positive longitudes and an Asia/Chongqing
time zone, despite being located in San Diego and Phoenix.

Using FAA information published by AirNav, I corrected their
coordinates in `airports_clean`:

- MYF: latitude 32.8157222, longitude -117.1395556.
- DVT: latitude 33.6883056, longitude -112.0825556.

I left their other fields unchanged. Their incorrect time-zone
fields are not used to determine the coordinate rankings.

Sources:
- https://www.airnav.com/airport/MYF
- https://www.airnav.com/airport/DVT

After correction, Eareckson Air Station (SYA) has the largest signed
longitude in the supplied dataset: 174.113620° E. Its positive
longitude is consistent with its location on Shemya Island, Alaska,
across the 180° meridian.

For this analysis, "easternmost" means the largest signed longitude
on the -180° to +180° scale. Under that definition, SYA is the answer
among airports in this dataset. This differs from the usual map
view of the contiguous United States, where EPM leads this ranking.

Location verification:
https://dec.alaska.gov/Applications/SPAR/PublicMVC/CSP/SiteReport/63

In [7]:
# Select February 12, 2013, using the dataset's local date fields.
weather_day = weather.loc[
    (weather["year"] == 2013)
    & (weather["month"] == 2)
    & (weather["day"] == 12)
].copy()

# Inspect the highest wind speeds before calculating rankings.
display(
    weather_day.nlargest(10, "wind_speed")[
        ["origin", "year", "month", "day", "hour",
         "wind_speed", "wind_gust"]
    ]
)

,origin,year,month,day,hour,wind_speed,wind_gust
1009,EWR,2013,2,12,3,1048.36058,NaN
18417,LGA,2013,2,12,2,23.01560,31.07106
1018,EWR,2013,2,12,12,21.86482,31.07106
18428,LGA,2013,2,12,13,21.86482,25.31716
1008,EWR,2013,2,12,2,20.71404,25.31716
9717,JFK,2013,2,12,8,20.71404,27.61872
18429,LGA,2013,2,12,14,20.71404,25.31716
9716,JFK,2013,2,12,7,19.56326,NaN
18416,LGA,2013,2,12,1,19.56326,28.76950
18427,LGA,2013,2,12,12,19.56326,26.46794


In [8]:
# Flag the specific implausible EWR reading.
outlier = (
    (weather_day["origin"] == "EWR")
    & (weather_day["hour"] == 3)
    & (weather_day["wind_speed"] > 1000)
)

# Preserve weather_day and mark only that wind-speed value as missing.
weather_day_clean = weather_day.copy()
weather_day_clean.loc[outlier, "wind_speed"] = float("nan")

print("Wind-speed readings excluded:", outlier.sum())

# Define "windiest" primarily by average hourly wind speed.
# Also compare medians, maximums, and valid observation counts.
wind_summary = (
    weather_day_clean.groupby("origin")["wind_speed"]
    .agg(["count", "mean", "median", "max"])
    .sort_values("mean", ascending=False)
)

display(wind_summary.round(2))

Wind-speed readings excluded: 1


,count,mean,median,max
origin,,,,
LGA,24,14.96,14.96,23.02
JFK,24,14.38,14.96,20.71
EWR,23,13.26,13.81,21.86


## Question 3: Windiest New York Area Airport

LaGuardia Airport (LGA) had the windiest weather on February 12,
2013, based on the highest average hourly wind speed after
excluding one implausible observation.

### Outlier Handling

EWR recorded a wind speed of 1,048.36058 mph at hour 3.
The next-highest observation that day was approximately 23.02 mph.
I treated the EWR reading as an invalid observation because it
is physically implausible for surface weather.

I preserved the original data and replaced only this wind-speed
value with NaN in `weather_day_clean`. I did not guess a replacement
value or remove the other measurements from that row.

### Results

| Airport | Valid readings | Mean wind speed (mph) | Maximum wind speed (mph) |
| --- | ---: | ---: | ---: |
| LGA | 24 | 14.96 | 23.02 |
| JFK | 24 | 14.38 | 20.71 |
| EWR | 23 | 13.26 | 21.86 |

### Assumptions

- "Windiest" means the highest mean of available hourly wind speeds.
- The maximum valid hourly wind speed also supports LGA as the answer.
- I selected the date using the dataset's year, month, and day fields.
- Missing wind speeds are excluded from the calculations, not treated
  as zero.
- EWR has one fewer valid reading after the outlier is excluded.
- Wind speed and wind gust are separate variables; this comparison
  uses wind speed.

Source for variable definitions and units:
https://nycflights13.tidyverse.org/reference/weather.html